In [ ]:
import numpy as np
import pandas as pd
from epiweeks import Week
import matplotlib.pyplot as plt
from scipy.stats import lognorm
from ensemble import compute_ensemble
from harmonize_preds import aux_functions
from cum_preds_method import estimate_rho_correlation, cumulative_estimation

In [2]:
import matplotlib as mpl 

mpl.rcParams['axes.edgecolor'] = 'gray'

# Definir a cor das linhas dos ticks maiores e menores como cinza
mpl.rcParams['xtick.color'] = 'gray'
mpl.rcParams['ytick.color'] = 'gray'
mpl.rcParams['xtick.labelcolor'] = 'black'
mpl.rcParams['ytick.labelcolor'] = 'black'
plt.rcParams['axes.labelsize'] = 16 # Axis labels
plt.rcParams['xtick.labelsize'] = 14  # X-axis tick labels
plt.rcParams['ytick.labelsize'] = 14  # Y-axis tick labels
plt.rcParams['font.size'] = 16  # General font size
FONT = 12


Load preds: 

In [3]:
df_for = pd.read_csv('predictions/forecasts_2nd_sprint_update.csv.gz')
df_for.date = pd.to_datetime(df_for.date)
df_for = df_for.loc[df_for.model_id != 137]
df_for.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,model_id,state
0,2025-10-05,0.0,0.0,0.0,101.922668,356.781097,790.528656,1992.101929,2496.932953,2749.348465,133,DF
1,2025-10-12,0.0,0.0,0.0,74.657440,267.429688,815.399750,2309.984863,2940.143433,3255.222717,133,DF
2,2025-10-19,0.0,0.0,0.0,4.947113,197.892944,957.998810,2695.268555,3423.885498,3788.193970,133,DF
3,2025-10-26,0.0,0.0,0.0,38.830750,305.259369,1268.370544,3319.793701,4166.516785,4589.878326,133,DF
4,2025-11-02,0.0,0.0,0.0,22.203766,362.994873,1516.120422,3958.920898,4977.201721,5486.342133,133,DF


In [4]:
df_for.model_id.unique().shape

(15,)

Load metrics: 

In [5]:
df_m_norm = pd.read_csv('results/metrics_wis_norm.csv.gz')

df_m_norm.head()

,model,state,validation_test,WIS
0,108,PA,1,0.306515
1,158,PA,1,0.235720
2,133,PA,1,0.178181
3,144,PA,1,0.162322
4,152,PA,1,0.628968


In [6]:
df_m_norm.model.unique().shape

(15,)

In [7]:
n_models = 5
best_models = {}

for state in df_m_norm.state.unique():

    best_models[f'{state}'] = df_m_norm.loc[(df_m_norm.state == state)].groupby(['state', 'model'])[['WIS']].mean().reset_index().sort_values(by = 'WIS').model.values[:n_models]


In [8]:
best_models['PR']

array([136, 144, 155, 154, 150])

Ensemble of the median of the top 5 models:

In [9]:
dfm_top = pd.DataFrame()

for st in df_for.state.unique(): 
    df_median_ens = compute_ensemble(df_for.loc[(df_for.state == st) &
                                                  (df_for.model_id.isin(best_models[st]))])
    df_median_ens.head()

    dfm_top = pd.concat([dfm_top, df_median_ens], ignore_index = True)

dfm_top['model_id'] = 'E_median_top'

dfm_top.head()

,date,state,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,model_id
0,2025-10-05,DF,1.0,2.0,4.0,9.0,36.316665,86.010745,157.0,257.874179,398.000000,E_median_top
1,2025-10-12,DF,2.0,2.0,4.0,10.0,34.649051,94.248957,196.0,292.000000,405.445650,E_median_top
2,2025-10-19,DF,2.0,3.0,5.0,12.0,35.146760,90.000000,180.0,283.101775,421.337491,E_median_top
3,2025-10-26,DF,2.0,3.0,6.0,14.0,40.399247,104.193800,205.0,286.000000,435.649320,E_median_top
4,2025-11-02,DF,3.0,4.0,7.0,16.0,63.000000,129.745682,270.0,358.107097,500.000000,E_median_top


In [10]:
df_for.loc[(df_for.state == st) &
        (df_for.model_id.isin(best_models[st]))].model_id.unique()

array([156, 158, 152, 154, 157])

In [11]:
dfm_top.state.value_counts()

state
DF    53
MA    53
PR    53
AL    53
TO    53
RN    53
AC    53
RR    53
PB    53
SE    53
AP    53
MS    53
CE    53
RO    53
PI    53
RS    53
GO    53
BA    53
SP    53
MG    53
PA    53
MT    53
SC    53
AM    53
RJ    53
PE    53
ES    53
Name: count, dtype: int64

## Cumulative cases

### Get log normal aproximation of the preds and apply the copula method:

In [12]:
l_pos = tuple([col for col in df_median_ens.columns if 'lower' in col])
u_pos = tuple([col for col in df_median_ens.columns if 'upper' in col])
the_pos = list(l_pos + u_pos)
l_pos = list(l_pos)
u_pos = list(u_pos)

lvls = (0.5, 0.8, 0.9, 0.95)
p = [aux_functions.quantile_pair(lvl) for lvl in lvls]
p = [i for quantile in p for i in quantile]
p.append(0.5)
ps = p.copy()
ps.sort()

In [13]:
def process_row(row):
    '''
    Estimate the lognormal parameters for each row in the dataframe
    '''
    #the_pos =  [ 'upper_50', 'upper_80', 'upper_90', 'upper_95']

    #ps = [0.5, 0.75,0.9,0.95, 0.975]
    # pegar valores da linha
    xi_values = list(row[the_pos].values) if hasattr(row[the_pos], "values") else [row[the_pos]]
    xi_values.append(row["pred"])
    
    # ordenar
    xi_sorted = sorted(xi_values)

    out = pd.DataFrame({
    'p': ps,
    'xi': xi_sorted
    })

    out = out.loc[out.xi > 0]
    # ajustar CDF (aqui você precisa garantir que 'out' está definido de fora
    cdf_fit = aux_functions.fit_ln_CDF(
        x=out["xi"],
        Fhat=out["p"],
        weighting=1
    )
    
    return cdf_fit

In [14]:
%%time 

dfm_top[['mu', 'sigma']] = dfm_top.apply(process_row, axis=1, result_type = 'expand')

df_median_ens = dfm_top

df_median_ens.head()

CPU times: user 35.4 s, sys: 107 ms, total: 35.5 s
Wall time: 35.7 s


,date,state,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,model_id,mu,sigma
0,2025-10-05,DF,1.0,2.0,4.0,9.0,36.316665,86.010745,157.0,257.874179,398.000000,E_median_top,3.108446,1.468397
1,2025-10-12,DF,2.0,2.0,4.0,10.0,34.649051,94.248957,196.0,292.000000,405.445650,E_median_top,3.349067,1.355086
2,2025-10-19,DF,2.0,3.0,5.0,12.0,35.146760,90.000000,180.0,283.101775,421.337491,E_median_top,3.368291,1.364894
3,2025-10-26,DF,2.0,3.0,6.0,14.0,40.399247,104.193800,205.0,286.000000,435.649320,E_median_top,3.384992,1.373416
4,2025-11-02,DF,3.0,4.0,7.0,16.0,63.000000,129.745682,270.0,358.107097,500.000000,E_median_top,3.656610,1.305125


In [15]:
import polars as pl
import scipy.stats as st
from tqdm import tqdm

In [16]:
df_median_ens.state.unique()

array(['DF', 'PI', 'PE', 'RJ', 'AM', 'SC', 'MT', 'PA', 'MG', 'SP', 'BA',
       'GO', 'RS', 'RO', 'MA', 'CE', 'MS', 'AP', 'SE', 'PB', 'RR', 'AC',
       'RN', 'TO', 'AL', 'PR', 'ES'], dtype=object)

In [ ]:
dengue = pl.read_csv('data/dengue_agg.csv.gz')
dengue = dengue.rename({'uf': 'state'})

dengue = dengue[['state','date','casos']]

dengue = dengue.with_columns(
    pl.col("date").str.strptime(pl.Date, format="%Y-%m-%d").alias("date")
)

dengue = dengue.with_columns(pl.col('date').dt.year().alias('year'))

dengue


state,date,casos,year
str,date,i64,i32
"""AC""",2010-01-03,760,2010
"""TO""",2010-01-03,231,2010
"""SP""",2010-01-03,1628,2010
"""SE""",2010-01-03,3,2010
"""SC""",2010-01-03,10,2010
…,…,…,…
"""AL""",2025-07-27,54,2025
"""AC""",2025-07-27,27,2025
"""SP""",2025-07-27,843,2025


In [18]:
F_marginals = pl.from_pandas(df_median_ens[['date', 
                                            'state',
                                            'mu','sigma']].sort_values(by ='date'))


F_marginals


date,state,mu,sigma
datetime[ns],str,f64,f64
2025-10-05 00:00:00,"""DF""",3.108446,1.468397
2025-10-05 00:00:00,"""AP""",2.345674,1.196794
2025-10-05 00:00:00,"""ES""",3.1336,1.598805
2025-10-05 00:00:00,"""PI""",3.112279,0.76677
2025-10-05 00:00:00,"""BA""",5.332973,0.541704
…,…,…,…
2026-10-04 00:00:00,"""RR""",1.168347,1.358979
2026-10-04 00:00:00,"""RJ""",4.592079,1.119501
2026-10-04 00:00:00,"""RS""",3.154959,1.609703


In [20]:
rhos = []
states = np.sort(dengue['state'].unique())
for state in states:
    history = dengue.filter(pl.col('state') == state).sort('date')['casos']
    rho = estimate_rho_correlation(history)
    rhos.append(rho)

rhos = pl.DataFrame({'state': states, 'rho': rhos})

In [21]:
rhos

state,rho
str,f64
"""AC""",0.945004
"""AL""",0.96378
"""AM""",0.92289
"""AP""",0.917369
"""BA""",0.966292
…,…
"""RS""",0.904208
"""SC""",0.912145
"""SE""",0.927776


In [23]:
states = rhos.sort('state')['state'].unique()
models = ["E_median_all"]
results = []

for state in tqdm(states, desc="States"):
    for model in tqdm(models, desc=f"Models for {state}", leave=False):
        rho = rhos.filter(pl.col('state') == state)['rho'].to_numpy()[0]
        forecast_marginals = []
        for row in F_marginals.filter(pl.col('state') == state).iter_rows(named=True):
            mu, sigma = row['mu'], row['sigma']
            forecast_marginals.append(st.lognorm(s=sigma, scale=np.exp(mu)))
        cumu = cumulative_estimation(forecast_marginals, rho, n_paths=2000)
        results.append({
            'state': state,
            'model': model.split('_')[1],
            'lower_90': cumu[0],
            'pred': cumu[1],
            'upper_90': cumu[2],
            'rho': rho
        })

results = pl.DataFrame(results)

States: 100%|██████████| 27/27 [07:05<00:00, 15.75s/it]


In [24]:
results

state,model,lower_90,pred,upper_90,rho
str,str,f64,f64,f64,f64
"""PE""","""median""",6114.135203,18690.127178,57541.416383,0.966105
"""CE""","""median""",3444.601541,10711.140588,35804.477678,0.972429
"""DF""","""median""",4463.947496,21725.280657,103594.774372,0.96051
"""RO""","""median""",1140.542373,3708.065937,13157.87635,0.925388
"""PA""","""median""",5328.694908,10384.027492,20640.464333,0.943659
…,…,…,…,…,…
"""RN""","""median""",4034.142331,12585.017448,44017.870312,0.963859
"""MT""","""median""",13196.358308,25514.985554,50727.630776,0.960568
"""RR""","""median""",152.243361,425.123673,1418.514626,0.891562


In [25]:
df_cum_cop = results.to_pandas()
df_cum_cop.head()

,state,model,lower_90,pred,upper_90,rho
0,PE,median,6114.135203,18690.127178,57541.416383,0.966105
1,CE,median,3444.601541,10711.140588,35804.477678,0.972429
2,DF,median,4463.947496,21725.280657,103594.774372,0.960510
3,RO,median,1140.542373,3708.065937,13157.876350,0.925388
4,PA,median,5328.694908,10384.027492,20640.464333,0.943659


save the results:

In [26]:
df_median_ens.to_csv('predictions/ensemble_median_2026.csv.gz', index = False)

In [27]:
df_cum_cop.to_csv('predictions/ensemble_median_2026_cum_cases.csv.gz', index = False)

### Compute the cumulative cases for the best model in the period:

In [28]:
df_best_st_model = df_m_norm.loc[df_m_norm.groupby("state")["WIS"].idxmin()][['model', 'state']].rename(columns = {'model':'model_id'})
df_best_st_model.head()

,model_id,state
1058,157,AC
1004,156,AL
925,150,AM
964,158,AP
883,145,BA


In [29]:
tuples_to_keep = tuples = list(df_best_st_model[['model_id', 'state']].itertuples(index=False, name=None))

df_for_best = df_for[df_for[['model_id', 'state']].apply(tuple, axis=1).isin(tuples_to_keep)]

df_for_best.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,model_id,state
795,2025-10-05,0.0,0.0,0.0,0.0,22.241272,57.594818,92.021179,104.038086,110.046539,133,CE
796,2025-10-12,0.0,0.0,0.0,0.0,18.411377,50.900024,80.947754,91.022369,96.059677,133,CE
797,2025-10-19,0.0,0.0,0.0,0.0,16.388184,43.789032,69.310181,77.532959,81.644348,133,CE
798,2025-10-26,0.0,0.0,0.0,0.0,23.394104,46.906128,81.508179,94.610596,101.161804,133,CE
799,2025-11-02,0.0,0.0,0.0,0.0,29.567505,47.183105,87.846558,104.565033,112.924271,133,CE


get the lognormal parameters:

In [30]:
df_for_best[['mu', 'sigma']] = df_for_best.apply(process_row, axis=1, result_type = 'expand')


/var/folders/ch/kxpr39wx44v97968yr_4hmch0000gn/T/ipykernel_10564/4271899542.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_for_best[['mu', 'sigma']] = df_for_best.apply(process_row, axis=1, result_type = 'expand')
/var/folders/ch/kxpr39wx44v97968yr_4hmch0000gn/T/ipykernel_10564/4271899542.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_for_best[['mu', 'sigma']] = df_for_best.apply(process_row, axis=1, result_type = 'expand')


In [31]:
F_marginals = pl.from_pandas(df_for_best[['date', 
                                            'state',
                                            'mu','sigma']].sort_values(by ='date'))


In [32]:
states = rhos.sort('state')['state'].unique()
results = []

for state in tqdm(states, desc="States"):
    
    rho = rhos.filter(pl.col('state') == state)['rho'].to_numpy()[0]
    forecast_marginals = []
    for row in F_marginals.filter(pl.col('state') == state).iter_rows(named=True):
        mu, sigma = row['mu'], row['sigma']
        forecast_marginals.append(st.lognorm(s=sigma, scale=np.exp(mu)))
    
    cumu = cumulative_estimation(forecast_marginals, rho, n_paths=2000)
    results.append({
            'state': state,
            'model': model.split('_')[1],
            'lower_90': cumu[0],
            'pred': cumu[1],
            'upper_90': cumu[2],
            'rho': rho
        })

results = pl.DataFrame(results)

States: 100%|██████████| 27/27 [06:56<00:00, 15.43s/it]


In [33]:
df_cum_best_models = results.to_pandas()

df_cum_best_models = df_cum_best_models[['state', 'lower_90', 'pred', 'upper_90']].merge(df_best_st_model, on = 'state')

df_cum_best_models.head()

,state,lower_90,pred,upper_90,model_id
0,RR,188.795619,470.924381,1.231674e+03,157
1,AP,220.629907,987.042713,5.113742e+03,158
2,SE,961.884927,1891.884242,4.108112e+03,145
3,RN,3186.258323,9983.876187,3.167938e+04,156
4,SP,360069.664901,635895.535254,1.160364e+06,155


save the results:

In [34]:
df_cum_best_models.to_csv('predictions/best_models_2026_cum_cases.csv.gz', index = False)